# NL2SQL Training & Evaluation Orchestrator
This notebook strictly orchestrates the `nl2sql` Python codebase. All logic resides under `src/nl2sql/`.

**Codebase:** https://github.com/Vtheonly/nl2sql_engine — Section 1 clones this repository directly, so the notebook runs end-to-end on a fresh Colab / Kaggle runtime.

In [ ]:
# Section 1: Environment & Codebase Setup (Fresh Colab / Kaggle Runtime)
# The notebook pulls the codebase directly from the GitHub repository, then installs it.
import os, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get("NL2SQL_REPO_URL", "https://github.com/Vtheonly/nl2sql_engine.git")
PROJECT_DIR = Path("/content/nl2sql_engine") if Path("/content").exists() else Path.cwd()

if REPO_URL and not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR / ".git").exists():
    # Re-running inside an existing checkout: fast-forward to the latest code.
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[constrained]"], check=True)
print("Working directory:", Path.cwd())

In [ ]:
# Section 2: Experiment & Training Configuration
from nl2sql.pipeline import NL2SQLPipeline

CONFIG_OVERRIDES = {
    "training": {
        "stage": "parser",          # "embedder" | "linker" | "parser" | "verifier" | "all"
        "epochs": 5,
        "batch_size": 8,
        "grad_accum_steps": 4,
        "learning_rate": 5e-5,
        "mixed_precision": "bf16",
        "resume_from": "auto",      # Automatically resumes interrupted training from latest checkpoint
        "output_dir": "./workspace/runs/exp_codet5p_220m",
    },
    "hub": {
        "enabled": False,
        "repo_id": None,            # e.g. "username/nl2sql-codet5p-220m"
        "push_checkpoints": False,
        "push_final_models": True,
        "push_eval_reports": True,
    },
}

pipeline = NL2SQLPipeline.from_config_file("configs/default_train.yaml", overrides=CONFIG_OVERRIDES)

In [ ]:
# Section 3: Acquire & Prepare Complete Datasets (Delegated to Codebase)
data_summary = pipeline.prepare_data()
data_summary

In [ ]:
# Section 4: Launch or Resume Training
train_results = pipeline.train()

In [ ]:
# Section 5: Display Training Curves & Resource Utilization
import pandas as pd

for stage_name, stage_res in train_results.items():
    print(f"=== Stage: {stage_name} | Best Val Loss: {stage_res['best_val_loss']:.4f} | Artifact: {stage_res['artifact_dir']} ===")
    df = pd.DataFrame(stage_res["history"])
    display(df[["epoch", "global_step", "train_loss", "val_loss", "lr", "is_best"]])

In [ ]:
# Section 6: Evaluate (Both Without and With Oracle Evidence) & Push Artifacts
metrics_no_ev = pipeline.evaluate(split="test", use_oracle_evidence=False)
metrics_with_ev = pipeline.evaluate(split="test", use_oracle_evidence=True)

print("Evaluation (No Oracle Evidence):", metrics_no_ev.model_dump())
print("Evaluation (With Oracle Evidence):", metrics_with_ev.model_dump())

hub_url = pipeline.push_artifacts()
print("Pushed artifacts URL:", hub_url)